# 번호판 **검출** 모델 학습 (YOLO)

**런타임 → 런타임 유형 변경 → T4 GPU**

## 왜 이걸 만드는가

실측 100장에서 오답 30건을 갈라 보니 이렇다.

    20건 (67%)   어디를 자를지 모름   ← 검출 문제
    10건 (33%)   무슨 글자인지 모름   ← 인식 문제

인식기는 이미 좋다 (문자 정확도 98.8%). **크롭만 제대로 주면 읽는다.**
문제는 번호판 위치를 EasyOCR 의 **범용 글자 검출**로 짐작한다는 것이다.
그건 화면의 모든 글자를 찾으므로 이런 일이 생긴다.

    253수3936  →  '253수28도33고3루3936885135'   주변 글자까지 긁어옴
    96우1967   →  '967'                          번호판 일부만 잡음

번호판만 찾도록 **전용 검출기**를 학습시키면 이 20건이 사라진다.

## 데이터

Roboflow `car-plate-data` 1,000장. 사람이 직접 그은 번호판 상자가 들어 있다.

한글 파일명이 내보내기 과정에서 날아가 **인식** 학습에는 못 썼지만,
**검출**에는 글자가 필요 없다. 상자 좌표만 있으면 된다.

    train 700장 / valid 200장 / test 100장
    클래스 1개: car_plate
    라이선스 CC BY 4.0


## 1. 환경

In [ ]:
!nvidia-smi -L || echo "GPU 없음 — 런타임 유형을 T4 GPU 로 바꿀 것"
!pip -q install ultralytics
from ultralytics import YOLO
import torch, os
print("CUDA:", torch.cuda.is_available())

## 2. 데이터 올리기

로컬에서 Roboflow 폴더를 통째로 압축한다. `d` 폴더 안에 `train`/`valid`/`test`
가 풀려 있으므로 그 셋을 선택해 압축하면 된다.

    C:\Users\박지원\Desktop\d
      train\  valid\  test\  data.yaml     ← 이 넷을 선택해 ZIP

용량이 작아서(수십 MB) 바로 올려도 된다.

In [ ]:
from google.colab import files
up = files.upload()                      # car_plate_data.zip
!unzip -q -o *.zip -d /content/plate_det
!ls /content/plate_det

In [ ]:
# data.yaml 의 경로를 Colab 기준으로 다시 쓴다.
# Roboflow 가 만든 파일은 '../train/images' 같은 상대경로라 그대로는 못 쓴다.
from pathlib import Path

root = Path('/content/plate_det')
if not (root / 'train').is_dir():                 # zip 안에 한 겹 더 있을 때
    inner = next(p for p in root.iterdir() if (p / 'train').is_dir())
    root = inner
print("데이터 위치:", root)

yaml_txt = f"""path: {root}
train: train/images
val: valid/images
test: test/images
nc: 1
names: ['car_plate']
"""
Path('/content/plate.yaml').write_text(yaml_txt)
print(yaml_txt)

for s in ('train', 'valid', 'test'):
    n = len(list((root / s / 'images').glob('*')))
    m = len(list((root / s / 'labels').glob('*.txt')))
    print(f"  {s}: 이미지 {n} / 라벨 {m}")

## 3. 학습

`yolo11n` (가장 작은 모델)로 충분하다. 찾을 것이 **번호판 한 종류**뿐이고
데이터가 1,000장이라, 큰 모델을 쓰면 오히려 과적합된다.

증강은 ultralytics 기본값을 쓰되 **좌우 반전은 끈다.** 번호판은 뒤집히지 않는다.

In [ ]:
model = YOLO('yolo11n.pt')          # COCO 사전학습에서 출발
results = model.train(
    data='/content/plate.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,                    # 20 에폭 좋아지지 않으면 멈춘다
    fliplr=0.0,                     # 좌우 반전 금지 (번호판은 뒤집히지 않는다)
    degrees=10, translate=0.1, scale=0.5, shear=3,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    project='/content/runs', name='plate', exist_ok=True,
)

## 4. 성능 확인

**mAP50** 을 본다. 상자가 정답과 50% 이상 겹치면 맞다고 보는 지표다.
번호판 검출은 쉬운 편이라 0.95 이상 나와야 정상이다.

In [ ]:
m = model.val(data='/content/plate.yaml', split='test')
print(f"\n  mAP50     {m.box.map50:.3f}   ← 0.95 이상이면 정상")
print(f"  mAP50-95  {m.box.map:.3f}")
print(f"  정밀도     {m.box.mp:.3f}")
print(f"  재현율     {m.box.mr:.3f}   ← 놓치는 번호판 비율 = 1 - 이 값")

눈으로도 확인한다. 상자가 번호판에 딱 맞는지 본다.

In [ ]:
import glob
from IPython.display import Image, display
best = '/content/runs/plate/weights/best.pt'
det = YOLO(best)
imgs = sorted(glob.glob(f'{root}/test/images/*'))[:8]
det.predict(imgs, save=True, project='/content/pred', name='v', exist_ok=True, conf=0.25)
for p in sorted(glob.glob('/content/pred/v/*'))[:8]:
    display(Image(p, width=460))

## 5. 내려받기

`plate_det.pt` 를 받아 `omeca-lpr` 폴더에 둔다.

In [ ]:
import shutil
shutil.copy('/content/runs/plate/weights/best.pt', '/content/plate_det.pt')
print("크기:", round(os.path.getsize('/content/plate_det.pt')/1024/1024, 1), "MB")
from google.colab import files
files.download('/content/plate_det.pt')

## 6. 내 PC 에서 확인

`plate_det.pt` 를 `C:\Users\박지원\Desktop\d\omeca-lpr\` 에 넣고,
**개발용 100장**으로 잰다.

```powershell
python bench_lpr.py --dir plates --dir plates_test --weights plate_det.pt
```

지금(검출기 없음)이 **42~66%** 였다. 검출 문제 20건이 사라지면
그만큼 오를 것이다. 안 오르면 `--weights` 를 빼면 원래대로 돌아간다.

> **`plates_final/` 로는 아직 재지 않는다.** 다 고친 뒤 딱 한 번이다.
